# Waveform plots — Task Difference Simple−Complex (Load paper, active only)

Plots the within-subject Simple−Complex task difference waveform.
Reads from `outputs/grand_averages/load/taskdiff/` pkls.

Confirmed plots:
- Target N2: central cluster (FC1,FCz,FC2,C1,Cz,C2), 180–230ms
- Novel N2:  central cluster (FC1,FCz,FC2,C1,Cz,C2), 150–220ms

Batch configs live in `config_load.BATCH_TASKDIFF`.

Last edit: 15.10.2026 @anguyen

In [ ]:
from __future__ import annotations
from datetime import datetime
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import mne
from mne.viz import set_browser_backend

def find_repo_root(marker='core') -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / marker).is_dir():
            return candidate
    raise RuntimeError(f"Could not locate repo root (no '{marker}/' found above {Path.cwd()})")

sys.path.insert(0, str(find_repo_root() / 'core'))
from config_load import BATCH_TASKDIFF, STYLES_TASKDIFF
from participants import get_participants
from utils import get_cluster_label, load_pkl, show_latest_pkls, tuple_to_string_fname, tuple_to_string_title

%matplotlib qt
set_browser_backend('qt')

plt.rcParams.update({
    'font.size':        12,
    'axes.titlesize':   13,
    'axes.labelsize':   12,
    'xtick.labelsize':  10,
    'ytick.labelsize':  10,
    'legend.fontsize':  10,
    'figure.titlesize': 16,
})

In [ ]:
# ==============================================================================
# Configuration — load paper, active only
# ==============================================================================

PAPER    = 'load'
PARADIGM = 'active'
MEASURE  = 'raw'
LOCK     = 'stim'

DATE_STR = datetime.today().strftime('%Y%m%d')

if sys.platform == 'win32':
    root = Path('L:/EEG_Flow_data/Study_EEG_Flow_Active/derivatives_v2')
else:
    root = Path('/Volumes/faculty/EEG_Flow_data/Study_EEG_Flow_Active/derivatives_v2')

input_path = root / 'outputs' / 'grand_averages' / PAPER / 'taskdiff'
root_img   = root / 'outputs' / 'img' / PAPER / 'taskdiff'
root_img.mkdir(parents=True, exist_ok=True)
(root_img / MEASURE).mkdir(parents=True, exist_ok=True)

n           = len(get_participants(PAPER, PARADIGM))
CONDITIONS  = {'Simple\u2212Complex': 'taskdiff'}

DATE_SOURCE = show_latest_pkls(root, PAPER, PARADIGM, LOCK, MEASURE, None, subfolder='taskdiff')

print(f'Input : {input_path}')
print(f'Output: {root_img}')

In [ ]:
# ==============================================================================
# Load pkl
# ==============================================================================

fname = f'{DATE_SOURCE}_{PARADIGM}_{n}p_taskdiff_stim_raw.pkl'
cond_key = 'Simple\u2212Complex'
evokeds  = {cond_key: load_pkl(input_path / fname)}
print(f'Loaded <- {fname}')
print(f'Keys: {list(evokeds[cond_key].keys())}')

In [ ]:
# ==============================================================================
# plot_compare_condition
# ==============================================================================

def plot_compare_condition(
    stim: str,
    component: str,
    cluster: list[str],
    windows: tuple,
    ylim: dict,
) -> None:
    evokeds_plot = {
        label: [e.copy().pick('eeg') for e in data[stim]]
        for label, data in evokeds.items()
        if stim in data
    }
    if not evokeds_plot:
        print(f"No data found for stim='{stim}' — skipping.")
        return

    if isinstance(windows[0], (int, float)):
        windows = (windows,)

    n_participants = len(next(iter(evokeds_plot.values())))
    txt_cluster    = get_cluster_label(cluster)
    fname_windows  = tuple_to_string_fname(windows)
    title_windows  = tuple_to_string_title(windows)

    fig, ax = plt.subplots(1, 1, figsize=(4.5, 4.5))
    mne.viz.plot_compare_evokeds(
        evokeds_plot,
        invert_y=True,
        ylim=ylim,
        axes=ax,
        vlines=[],
        truncate_yaxis=False,
        truncate_xaxis=False,
        picks=cluster,
        combine='mean',
        title='',
        styles=STYLES_TASKDIFF,
        time_unit='ms',
    )

    ax.axvline(0, color='black', linestyle='solid', linewidth=1.2)
    ax.axhline(0, color='black', linestyle='solid', linewidth=1.2)
    for start, end in windows:
        ax.axvspan(start, end, facecolor='#ffbd66', alpha=0.3)

    ax.xaxis.set_major_locator(ticker.MultipleLocator(100))
    ax.yaxis.set_major_locator(ticker.MultipleLocator(2))
    ax.grid(color='gray', linestyle='--', linewidth=0.5, alpha=0.3)
    ax.set_ylabel('Amplitude (\u00b5V)')
    ax.legend(loc='lower left')
    plt.subplots_adjust(left=0.05, right=0.95)

    fig.suptitle(
        f'[{PARADIGM.capitalize()}] {stim.capitalize()} \u00b7 {component} \u2014 Task Diff'
    )
    fig.text(
        0.5, 0.895,
        f'{title_windows} \u00b7 {txt_cluster} cluster \u00b7 N={n_participants}',
        ha='center', fontsize=12
    )

    out_fname = (
        root_img / MEASURE
        / f'{PARADIGM}_{n_participants}p_{PAPER}_{MEASURE}_taskdiff_wave'
          f'_{stim}_{component}_{txt_cluster}_{fname_windows}_{DATE_STR}.svg'
    )
    print(out_fname)
    plt.savefig(out_fname, dpi=300, bbox_inches='tight')
    plt.show()

print('plot_compare_condition() ready.')

## Batch run

Runs all pre-configured plots. Edit configs in `config_load.BATCH_TASKDIFF`.

In [ ]:
batch_configs = BATCH_TASKDIFF.get((PARADIGM, LOCK), [])

if batch_configs:
    print(f'Running {len(batch_configs)} plots [taskdiff]...\n')
    for cfg in batch_configs:
        plot_compare_condition(**cfg)
else:
    print('No BATCH_TASKDIFF configs found — add entries to config_load.BATCH_TASKDIFF.')

## Live / custom plots

In [ ]:
# plot_compare_condition(
#     stim      = 'target',
#     component = 'N2',
#     cluster   = ['FC1', 'FCz', 'FC2', 'C1', 'Cz', 'C2'],
#     windows   = ((180, 230),),
#     ylim      = dict(eeg=[-6, 8]),
# )